In [6]:
import sys 
sys.path.append(r"C:\Users\g0004878\Desktop\My codes\Frequently used codes\Frequently-used-codes")
sys.path.append(r"C:\Users\g0004878\Desktop\Projects in FY25-26\utils_files")
import Snowflake_configuration
snowflake_conn_prop = Snowflake_configuration.ds1_role_json
from snowflake.snowpark.session import Session
import pandas as pd
pd.set_option('display.max_columns',None)

import snowflake.snowpark as snowpark
from snowflake.snowpark import functions as F
from snowflake.snowpark.window import Window
import datetime

### Activate the session

In [7]:
session = Session.builder.configs(snowflake_conn_prop).create()

### Config_variables

### Step 1

In [5]:
# Planning months to compute SOQ for
MONTHS = ['2026-06-01']

# Stock snapshot timing — 'first' (start of month) or 'mid' (mid-month)
STOCK_DATE_TYPE = ['first']

# Increment each time the pipeline is re-run — used to version output rows
RUN_VERSION = 33

# Auto-derived today's date — used as a partition key in output tables
RUN_DATE = datetime.datetime.today().strftime('%Y%m%d')

# ABC classification → safety stock days
# A = high-revenue families (30 days stock)
# B = mid-tier families     (25 days stock)
# C = tail families         (20 days stock)
ABC = {'A': 30, 'B': 25, 'C': 20}

# Z-scores per service level
# Controls how many standard deviations of demand variability safety stock covers
# e.g. 95% service level → Z = 1.65 → covers 95% of demand spike scenarios
Z_SCORE = {95: 1.65, 90: 1.28, 85: 1.04, 80: 0.85, 99: 2.33}

# Human-readable string of ABC config — stored in output rows for auditability
STR_ABC = ', '.join([f'{k}:{v}' for k, v in ABC.items()])

# OBD flag — whether this run uses OBD (On-Board Diagnostics) mapped SKUs
IS_OBD   = True
OBD_FLAG = 'Y' if IS_OBD else 'N'

### Step 2

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — Table references
# All source and destination table names centralised here.
# If a table is renamed or moved, change it once here — not inside functions.
# ─────────────────────────────────────────────────────────────────────────────

# Input: pre-computed demand forecasts + stock positions per dealer-family-SKU
BASE_SOQ_TABLE = 'MOP_DATABASE.SOQ.SOQ_BASE_TABLE_FINAL_CONCATENATED'

# Input: demand variability at SKU level (how much each SKU's sales fluctuate)
DEMAND_VARIABILITY_TABLE = 'MOP_DATABASE.SOQ.DEMAND_VARIABILITY_SKU_FINAL_VERSION'

# Input: demand variability at model family level (all SKUs in a family share one value)
DEMAND_VARIABILITY_FAM_TABLE = 'MOP_DATABASE.SOQ.DEMAND_VARIABILITY_SKU_MODEL_FAMILY_FINAL_VERSION'

# Output: final SOQ results — appended per run, not overwritten
SOQ_OUTPUT_TABLE = 'MOP_DATABASE.SOQ.SOQ_DATA_FINAL_VERSION_2'

# Output: rows dropped due to missing lead time — saved for investigation
NULL_LEAD_TIME_TABLE = 'MOP_DATABASE.SOQ.NULL_LEAD_TIME'

# Input: end-of-journey (discontinuation) recommendations per SKU
END_JOURNEY_TABLE = 'MOP_DATABASE.SOQ.END_OF_JOURNEY_RECOMMENDATION'

In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — ABC helper function
# Pure Python — no Snowpark dependencies inside.
# Registered as a UDF in Step 7 so it executes server-side in Snowflake.
#
# Classifies a dealer-family based on its cumulative % of dealer's total sales:
#   cumulative < 70%  → A  (top revenue families — largest safety stock)
#   cumulative 70–90% → B  (mid-tier)
#   cumulative > 90%  → C  (tail families — smallest safety stock)
# ─────────────────────────────────────────────────────────────────────────────
def _get_abc_class(cumulative_pct: float) -> str:
    if cumulative_pct < 70:
        return 'A'
    elif cumulative_pct > 90:
        return 'C'
    else:
        return 'B'